In [1]:
pip install FlagEmbedding

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.6 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=4b55737afadcc7a8dceeaa26055b1a9afc968ab5ef688bb16c4f4e295340ff3e
  Stored in directory: /root/.cache/pip/wheels/f6/85/c2/9f0f621def52a1d5db7d29984f81e45f9fb6dfeb1a4eb6e31c
  Created wheel for cbor: filename=cbor-1.0.0-cp312-cp312-linux_x86_64.whl size=55022 sha256=6be9081edf8482b14542421599b7d30d036b007671574acde142293c78b97afa
 

In [2]:
# %% [markdown]
# # BGE-M3 bi-encoder rerank with score threshold

# %% [code]
import json
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from datasets import load_dataset

#với mỗi question -> 4000 articles lquan -> recall 92%
#bi-encoder 30-50 articles (recall ~80-85%) -> 30-50 vào trong qwen 
# %% [code]
DATASET_NAME = "tmquan/phapdien-moj-gov-vn"
DATASET_CONFIG = "articles"
DATASET_SPLIT = "train"
MODEL_NAME = "AITeamVN/Vietnamese_Embedding"

INPUT_PATH = Path("/kaggle/input/datasets/trnhuhuyhonggg/r2ai-top-bm25/submission.json")
OUTPUT_PATH = Path("bge_m3_biencoder_threshold_submission.json")

SCORE_THRESHOLD = 0
TOP_K = -1
BATCH_SIZE = 32
MAX_LENGTH = 1024
USE_FP16 = True

ARTICLE_RE = re.compile(r"Điều\s+([0-9]+(?:\.[0-9]+)*)", re.IGNORECASE)
DOC_RE = re.compile(
    r"\b(Bộ luật|Luật|Nghị định|Thông tư liên tịch|Thông tư|Quyết định|"
    r"Nghị quyết|Pháp lệnh|Lệnh|Chỉ thị)\s+(?:số\s+)?([0-9][^\s,;)]+)",
    re.IGNORECASE,
)


# %% [code]
@dataclass
class LawArticle:
    content: str
    doc_code: str
    doc_name: str
    article_label: str

    @property
    def lookup_key(self):
        return f"{self.doc_code}|{self.article_label}".lower()

    @property
    def relevant_doc(self):
        return f"{self.doc_code}|{self.doc_name}"

    @property
    def relevant_article(self):
        return f"{self.doc_code}|{self.doc_name}|{self.article_label}"


# %% [code]
def read_json(path):
    with path.open("r", encoding="utf-8-sig") as file:
        return json.load(file)


def write_json(path, data):
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)


def parse_candidate(candidate):
    parts = str(candidate).split("|")
    doc_code = parts[0].strip() if len(parts) > 0 else ""
    article_label = parts[2].strip() if len(parts) > 2 else ""
    return doc_code, article_label


def candidate_key(doc_code, article_label):
    return f"{doc_code}|{article_label}".lower()


# %% [code]
def get_article_label(row):
    text = " ".join(
        [
            str(row.get("source_note_text") or ""),
            str(row.get("article_title") or ""),
        ]
    )

    match = ARTICLE_RE.search(text)
    if match:
        return f"Điều {match.group(1)}"

    return str(row.get("article_title") or "").split(".", 1)[0].strip()


def get_doc_info(row):
    source_note = str(row.get("source_note_text") or "")
    match = DOC_RE.search(source_note)

    if not match:
        return "", "Văn bản pháp luật"

    doc_code = match.group(2).strip(".,;)")
    doc_name = source_note[match.start() :].strip(" .,-;()")

    return doc_code, doc_name


def build_corpus(dataset):
    corpus = []

    for row in dataset:
        content = str(row.get("content_text") or "").strip()
        if not content:
            continue

        doc_code, doc_name = get_doc_info(row)

        corpus.append(
            LawArticle(
                content=content,
                doc_code=doc_code,
                doc_name=doc_name,
                article_label=get_article_label(row),
            )
        )

    return corpus


def build_article_lookup(corpus):
    lookup = {}

    for article in corpus:
        if article.doc_code and article.article_label:
            lookup.setdefault(article.lookup_key, article)

    return lookup


# %% [code]
def collect_candidate_articles(input_items, article_lookup):
    all_candidates = []
    missing = 0

    for item in input_items:
        candidates = []

        for candidate in item.get("relevant_articles", []):
            doc_code, article_label = parse_candidate(candidate)
            article = article_lookup.get(candidate_key(doc_code, article_label))

            if article is None:
                missing += 1
                continue

            candidates.append(article)

        all_candidates.append(candidates)

    return all_candidates, missing


# %% [code]
def load_embedding_model(model_name, use_fp16=True):
    from FlagEmbedding import BGEM3FlagModel

    return BGEM3FlagModel(model_name, use_fp16=use_fp16)


def batched(values, batch_size):
    for start in range(0, len(values), batch_size):
        yield values[start : start + batch_size]


def extract_dense_vectors(encoded):
    if isinstance(encoded, dict):
        return np.asarray(encoded["dense_vecs"], dtype=np.float32)
    return np.asarray(encoded, dtype=np.float32)


def normalize_rows(matrix):
    matrix = np.asarray(matrix, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return matrix / norms


def encode_texts(model, texts, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    vectors = []

    for batch in batched(texts, batch_size):
        encoded = model.encode(batch, batch_size=batch_size, max_length=max_length)
        vectors.append(extract_dense_vectors(encoded))

    if not vectors:
        return np.empty((0, 0), dtype=np.float32)

    return normalize_rows(np.vstack(vectors))


# %% [code]
def make_answer(articles):
    if not articles:
        return "Không tìm thấy điều luật liên quan."

    answer = " ".join(
        article.content.replace("\n", " ").strip()[:700]
        for article in articles[:3]
    )

    return answer[:1000]


def unique_relevant_docs(articles):
    seen = set()
    relevant_docs = []

    for article in articles:
        key = (article.doc_code, article.doc_name)

        if not article.doc_code or key in seen:
            continue

        relevant_docs.append(article.relevant_doc)
        seen.add(key)

    return relevant_docs


def relevant_articles(articles):
    return [
        article.relevant_article
        for article in articles
        if article.doc_code
    ]


def min_max_normalize_ranked(ranked):
    if not ranked:
        return []

    scores = np.asarray([score for article, score in ranked], dtype=np.float32)
    min_score = float(scores.min())
    max_score = float(scores.max())

    if max_score == min_score:
        return [
            (article, 1.0)
            for article, score in ranked
        ]

    return [
        (article, float((score - min_score) / (max_score - min_score)))
        for article, score in ranked
    ]


def apply_threshold_and_top_k(ranked, score_threshold, top_k):
    ranked = [
        (article, score)
        for article, score in ranked
        if score >= score_threshold
    ]

    if top_k != -1:
        ranked = ranked[:top_k]

    return ranked


def rank_with_biencoder(
    input_items,
    candidate_articles_by_item,
    model,
    score_threshold=SCORE_THRESHOLD,
    top_k=TOP_K,
):
    unique_articles = {}

    for candidates in candidate_articles_by_item:
        for article in candidates:
            unique_articles.setdefault(article.lookup_key, article)

    articles = list(unique_articles.values())
    print("Unique candidate articles:", len(articles))

    article_vectors = encode_texts(
        model=model,
        texts=[article.content for article in articles],
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
    )
    article_vector_by_key = {
        article.lookup_key: article_vectors[index]
        for index, article in enumerate(articles)
    }

    question_vectors = encode_texts(
        model=model,
        texts=[str(item.get("question") or "") for item in input_items],
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
    )

    output = []

    for item_index, (item, candidates) in enumerate(zip(input_items, candidate_articles_by_item), start=1):
        question_vector = question_vectors[item_index - 1]
        ranked = []

        for article in candidates:
            score = float(np.dot(question_vector, article_vector_by_key[article.lookup_key]))
            ranked.append((article, score))

        ranked.sort(key=lambda row: row[1], reverse=True)
        ranked = min_max_normalize_ranked(ranked)
        ranked.sort(key=lambda row: row[1], reverse=True)
        ranked = apply_threshold_and_top_k(
            ranked=ranked,
            score_threshold=score_threshold,
            top_k=top_k,
        )
        top_articles = [article for article, score in ranked]

        output.append(
            {
                "id": item.get("id"),
                "question": str(item.get("question") or ""),
                "answer": make_answer(top_articles),
                "relevant_docs": unique_relevant_docs(top_articles),
                "relevant_articles": relevant_articles(top_articles),
                "scores": [
                    {
                        "article": article.relevant_article,
                        "score": round(score, 6),
                    }
                    for article, score in ranked
                ],
            }
        )

        if item_index % 100 == 0:
            print(f"Ranked {item_index}/{len(input_items)} questions")

    return output


# %% [markdown]
# # LOAD BM25 SUBMISSION

# %% [code]
input_items = read_json(INPUT_PATH)

print("Questions:", len(input_items))


# %% [markdown]
# # LOAD CORPUS

# %% [code]
dataset = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
    split=DATASET_SPLIT,
)

corpus = build_corpus(dataset)
article_lookup = build_article_lookup(corpus)

print("Corpus articles:", len(corpus))
print("Lookup keys:", len(article_lookup))


# %% [markdown]
# # MAP CANDIDATES FROM 4000 FILE

# %% [code]
candidate_articles_by_item, missing_candidates = collect_candidate_articles(
    input_items=input_items,
    article_lookup=article_lookup,
)

print("Candidate lists:", len(candidate_articles_by_item))
print("Missing candidates:", missing_candidates)


# %% [markdown]
# # LOAD BI-ENCODER MODEL

# %% [code]
model = load_embedding_model(
    MODEL_NAME,
    use_fp16=USE_FP16,
)

print("Bi-encoder ready:", MODEL_NAME)


# %% [markdown]
# # RANK CANDIDATES

# %% [code]
submission = rank_with_biencoder(
    input_items=input_items,
    candidate_articles_by_item=candidate_articles_by_item,
    model=model,
    score_threshold=SCORE_THRESHOLD,
    top_k=TOP_K,
)

print("Submission rows:", len(submission))
submission[0]


# %% [markdown]
# # WRITE OUTPUT

# %% [code]
write_json(OUTPUT_PATH, submission)

print("Wrote:", OUTPUT_PATH)

Questions: 2000


README.md: 0.00B [00:00, ?B/s]

articles-00000-of-00007.parquet:   0%|          | 0.00/6.07M [00:00<?, ?B/s]

articles-00001-of-00007.parquet:   0%|          | 0.00/6.56M [00:00<?, ?B/s]

articles-00002-of-00007.parquet:   0%|          | 0.00/5.84M [00:00<?, ?B/s]

...

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Bi-encoder ready: AITeamVN/Vietnamese_Embedding
Unique candidate articles: 57520



initial target device: 100%|██████████| 2/2 [00:20<00:00, 10.04s/it]

Chunks: 100%|██████████| 2/2 [00:03<00:00,  1.66s/it]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.56it/s]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.54it/s]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.50it/s]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.49it/s]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.29it/s]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.50it/s]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.51it/s]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.43it/s]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.43it/s]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.46it/s]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.47it/s]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.46it/s]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.46it/s]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.47it/s]

Chunks: 100%|██████████| 2/2 [00:12<00:00,  6.13s/it]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.38i

Ranked 100/2000 questions
Ranked 200/2000 questions
Ranked 300/2000 questions
Ranked 400/2000 questions
Ranked 500/2000 questions
Ranked 600/2000 questions
Ranked 700/2000 questions
Ranked 800/2000 questions
Ranked 900/2000 questions
Ranked 1000/2000 questions
Ranked 1100/2000 questions
Ranked 1200/2000 questions
Ranked 1300/2000 questions
Ranked 1400/2000 questions
Ranked 1500/2000 questions
Ranked 1600/2000 questions
Ranked 1700/2000 questions
Ranked 1800/2000 questions
Ranked 1900/2000 questions
Ranked 2000/2000 questions
Submission rows: 2000
Wrote: bge_m3_biencoder_threshold_submission.json
